In [3]:
!pip install catboost -q
!pip install seaborn
!pip install xgboost
!pip install lightgbm


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
import pickle

#print(sklearn.__version__)

In [5]:
df = pd.read_csv('cleaned_data.csv')

print(f"Shape: {df.shape}")
print(f"Seasons: {df['Year'].min()} - {df['Year'].max()}")
print(f"Players: {df['Player'].nunique()}")
print(f"MVP candidates (Pts Won > 0): {(df['Pts Won'] > 0).sum()}")
print(f"Seasons: {df['Year'].min():.0f} - {df['Year'].max():.0f} ({df['Year'].nunique()} seasons)")
print(f"Unique players: {df['Player'].nunique()}")
print(f"Unique teams: {df['Team'].nunique()}")
print(f"Avg players per season: {df.groupby('Year').size().mean():.0f}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
df.head()
df.describe()

Shape: (17976, 45)
Seasons: 1980.0 - 2021.0
Players: 3275
MVP candidates (Pts Won > 0): 697
Seasons: 1980 - 2021 (42 seasons)
Unique players: 3275
Unique teams: 39
Avg players per season: 428
Missing values: 0

Column types:
float64    33
int64       6
str         6
Name: count, dtype: int64


,Unnamed: 0,Wt,Age,G,GS,MP,FG,FGA,FG%,3P,...,Pts Won,Pts Max,Share,W,L,W/L%,GB,PS/G,PA/G,SRS
count,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,...,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000,17976.000000
mean,8987.500000,215.879395,26.678961,54.055685,27.303238,20.496584,3.207682,7.033600,0.441621,0.446495,...,6.388679,39.745160,0.006065,39.731809,40.399366,0.495715,15.840065,102.351307,102.489047,-0.132477
std,5189.368555,27.289579,4.089597,25.253982,29.975434,10.100926,2.286849,4.713654,0.094456,0.634589,...,63.918459,202.645066,0.058911,12.816527,12.783673,0.154865,13.047137,7.262327,7.302082,4.511820
min,0.000000,133.000000,18.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,7.000000,9.000000,0.106000,0.000000,81.900000,83.400000,-14.680000
25%,4493.750000,195.000000,23.000000,35.000000,1.000000,12.100000,1.400000,3.300000,0.405000,0.000000,...,0.000000,0.000000,0.000000,30.000000,31.000000,0.378000,4.000000,96.900000,97.100000,-3.260000
50%,8987.500000,215.000000,26.000000,62.000000,12.000000,20.000000,2.700000,5.900000,0.446000,0.100000,...,0.000000,0.000000,0.000000,41.000000,40.000000,0.512000,14.000000,101.800000,102.500000,-0.010000
75%,13481.250000,235.000000,29.000000,77.000000,55.000000,29.000000,4.600000,9.900000,0.488000,0.700000,...,0.000000,0.000000,0.000000,50.000000,50.000000,0.610000,25.000000,107.700000,107.600000,3.130000
max,17975.000000,360.000000,44.000000,85.000000,84.000000,43.700000,13.400000,27.800000,1.000000,5.300000,...,1310.000000,1310.000000,1.000000,73.000000,72.000000,0.890000,56.000000,126.500000,130.800000,11.800000


In [6]:
df['PPG'] = df['PTS']
df['RPG'] = df['TRB']
df['APG'] = df['AST']
df['SPG'] = df['STL']
df['BPG'] = df['BLK']
df['MPG'] = df['MP']
df['TPG'] = df['TOV']

# Advanced metrics (still need to calculate these)
df['TS%'] = df['PTS'] / (2 * (df['FGA'] + 0.44 * df['FTA'])).replace(0, 1)
df['Usage'] = (df['FGA'] + 0.44 * df['FTA'] + df['TOV']) / df['MP'].replace(0, 1) * 100
df['Win_Rate'] = df['W'] / (df['W'] + df['L']).replace(0, 1)

# Extract primary position only (before hyphen)
df['Primary_Pos'] = df['Pos'].str.split('-').str[0]

# One-hot encode primary position
pos_dummies = pd.get_dummies(df['Primary_Pos'], prefix='Pos')
df = pd.concat([df, pos_dummies], axis=1)

print(f"Total columns: {df.shape[1]}")

Total columns: 61


In [7]:
seasons = sorted(df['Year'].unique())
split_idx = int(len(seasons) * 0.9)

train_seasons = seasons[:split_idx]
test_seasons = seasons[split_idx:]

train_df = df[df['Year'].isin(train_seasons)]
test_df = df[df['Year'].isin(test_seasons)]

print(f"Training seasons ({len(train_seasons)}): {train_seasons[0]}-{train_seasons[-1]}")
print(f"Test seasons ({len(test_seasons)}): {test_seasons[0]}-{test_seasons[-1]}")
print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")

Training seasons (37): 1980.0-2016.0
Test seasons (5): 2017.0-2021.0
Train size: 15346 | Test size: 2630


In [8]:
# PREPROCESSING - Define high-quality features only
selected_features = [
    # Team Success (MOST IMPORTANT for MVP)
    'W', 'Win_Rate', 'PS/G', 'PA/G',

    # Per-Game Stats (instead of totals)
    'PPG', 'RPG', 'APG', 'SPG', 'BPG', 'MPG',

    # Efficiency Metrics
    'FG%', '3P%', 'eFG%', 'FT%', 'TS%',

    # Advanced Metrics
    'Usage',

    # Availability (MVP needs to play)
    'G',

    # Player Attributes
    'Age'
]

# Filter to only features that exist in dataframe
feature_cols = [col for col in selected_features if col in df.columns]

# Prepare data
X_train = train_df[feature_cols].replace([np.inf, -np.inf], 0).fillna(0)
y_train = train_df['Pts Won']
X_test = test_df[feature_cols].replace([np.inf, -np.inf], 0).fillna(0)
y_test = test_df['Pts Won']

print(f"Selected features: {len(feature_cols)}")
X_train.head()

Selected features: 18


,W,Win_Rate,PS/G,PA/G,PPG,RPG,APG,SPG,BPG,MPG,FG%,3P%,eFG%,FT%,TS%,Usage,G,Age
0,63,0.768293,114.7,106.0,3.1,2.1,0.3,0.1,0.3,6.7,0.474,0.000,0.474,0.568,0.493631,54.328358,43,22
1,63,0.768293,114.7,106.0,11.1,2.6,3.6,0.8,0.2,21.4,0.472,0.406,0.543,0.826,0.575249,51.158879,80,31
2,63,0.768293,114.7,106.0,5.1,3.6,0.5,0.3,0.2,14.7,0.488,0.000,0.488,0.733,0.550043,35.619048,53,25
3,63,0.768293,114.7,106.0,2.2,2.8,0.3,0.1,0.9,11.1,0.393,0.000,0.393,0.786,0.446429,24.900901,67,34
4,63,0.768293,114.7,106.0,13.0,2.5,1.8,1.1,0.0,20.9,0.468,0.306,0.474,0.915,0.507654,67.004785,71,36


In [9]:
# ALGORITHM DEVELOPMENT
models = {}
predictions = {}

print("Training Random Forest...")
rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
models['Random Forest'] = rf
predictions['Random Forest'] = rf.predict(X_test)

print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=900,
    max_depth=7,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    colsample_bylevel=0.8,
    min_child_weight=5,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
models['XGBoost'] = xgb_model
predictions['XGBoost'] = xgb_model.predict(X_test)

print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=700,
    max_depth=10,
    learning_rate=0.07,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.85,
    subsample_freq=5,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.5,
    min_split_gain=0.01,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
models['LightGBM'] = lgb_model
predictions['LightGBM'] = lgb_model.predict(X_test)

print("Training CatBoost...")
cat_model = CatBoostRegressor(
    iterations=700,
    depth=8,
    learning_rate=0.08,
    l2_leaf_reg=2.0,
    bootstrap_type='Bernoulli',
    subsample=0.85,
    random_strength=0.5,
    border_count=128,
    task_type='CPU',
    random_state=42,
    verbose=False
)
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))
models['CatBoost'] = cat_model
predictions['CatBoost'] = cat_model.predict(X_test)

print("Training Gradient Boosting...")
gb = GradientBoostingRegressor(
    n_estimators=700,
    max_depth=7,
    learning_rate=0.07,
    min_samples_split=15,
    min_samples_leaf=6,
    subsample=0.85,
    max_features='sqrt',
    random_state=42
)
gb.fit(X_train, y_train)
models['Gradient Boosting'] = gb
predictions['Gradient Boosting'] = gb.predict(X_test)


Training Random Forest...
Training XGBoost...
Training LightGBM...
Training CatBoost...
Training Gradient Boosting...


In [10]:
import pickle
import os

os.makedirs('saved_models', exist_ok=True)
for name, model in models.items():
    filename = f"saved_models/{name.lower().replace(' ', '_')}.pkl"
    with open(filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"Saved {name} to {filename}")

pickle.dump(feature_cols, open('saved_models/feature_cols.pkl', 'wb'))

Saved Random Forest to saved_models/random_forest.pkl
Saved XGBoost to saved_models/xgboost.pkl
Saved LightGBM to saved_models/lightgbm.pkl
Saved CatBoost to saved_models/catboost.pkl
Saved Gradient Boosting to saved_models/gradient_boosting.pkl


In [11]:
# EVALUATION
results = []

for name, model in models.items():
    y_pred = predictions[name]

    # Test metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })

    print(f"{name:20s} | RMSE: {rmse:6.2f} | MAE: {mae:6.2f} | R²: {r2:.4f}")

results_df = pd.DataFrame(results).sort_values('RMSE')
print("\nBEST MODEL:", results_df.iloc[0]['Model'])

Random Forest        | RMSE:  39.99 | MAE:   6.32 | R²: 0.5013
XGBoost              | RMSE:  43.85 | MAE:   6.03 | R²: 0.4003
LightGBM             | RMSE:  42.87 | MAE:   7.84 | R²: 0.4269
CatBoost             | RMSE:  42.88 | MAE:   6.41 | R²: 0.4264
Gradient Boosting    | RMSE:  41.35 | MAE:   6.77 | R²: 0.4666

BEST MODEL: Random Forest
